In [ ]:
# Simulate on real quantum computer
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager # Thêm thư viện dịch mạch
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

service = QiskitRuntimeService(name="trandon")

# 1. Tạo mạch lượng tử gốc (Ý tưởng của bạn)
qc = QuantumCircuit(2, 2)
qc.h(0)          
qc.cx(0, 1)      
qc.measure([0, 1], [0, 1]) 

# 2. Tìm máy lượng tử thật đang rảnh nhất
print("\nSearching for the most optimal real quantum device...")
backend = service.least_busy(simulator=False, operational=True)
print("=" * 50)
print(f"Running on real quantum hardware: {backend.name}")
print("=" * 50)
# === BƯỚC SỬA LỖI: TRANSPILE (DỊCH MẠCH) ===
print("\nTranslating the circuit for the real quantum device...")
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(qc) # Đây là mạch đã được tối ưu cho máy thật
print("Translating success")
# ============================================

# 3. Gửi mạch ĐÃ DỊCH (isa_circuit) lên máy thật
print("\nSending to IBM Quantum...")

sampler = SamplerV2(mode=backend) 
job = sampler.run([isa_circuit], shots=1024) # Gửi isa_circuit thay vì qc

print(f"Job ID: {job.job_id()}")

result = job.result()
pub_result = result[0]

# 4. In kết quả
counts = pub_result.data.c.get_counts()
print("\n===== RESULTS FROM REAL QUANTUM HARDWARE =====")
print(counts)

qiskit_runtime_service.__init__:WARNING:2026-07-13 21:53:32,947: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().



Searching for the most optimal real quantum device...


qiskit_runtime_service.backends:WARNING:2026-07-13 21:53:33,444: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-07-13 21:53:37,072: Using instance: open-instance, plan: open


Running on real quantum hardware: ibm_marrakesh

Translating the circuit for the real quantum device...
Translating success

Sending to IBM Quantum...
Job ID: d9afnsug26ic73dekf30

===== RESULTS FROM REAL QUANTUM HARDWARE =====
{'11': 478, '00': 515, '10': 16, '01': 15}


In [ ]:
# Simulate Wigner's Friend experiment on real quantum computer
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

# Khởi tạo dịch vụ IBM Quantum từ profile của bạn
service = QiskitRuntimeService(name="trandon")

# =====================================================================
# 1. TẠO MẠCH LƯỢNG TỬ MÔ PHỎNG THÍ NGHIỆM WIGNER
# Chúng ta cần 3 Qubit đại diện cho: [0]: Hệ S, [1]: Friend, [2]: Wigner
# Và 2 bit cổ điển để lưu kết quả đo của Friend và Wigner ở cuối thí nghiệm
# =====================================================================
qc = QuantumCircuit(3, 2)

# --- Bước A: Hệ S được đưa vào trạng thái chồng chập ---
qc.h(0) 

# --- Bước B: Friend thực hiện phép đo lên Hệ S ---
# Trong cơ học lượng tử, việc một thực thể đo một hệ thống mà không phá hủy 
# tính mạch lạc đối với người quan sát bên ngoài được mô phỏng bằng cổng CNOT (CX).
qc.cx(0, 1) 

# Lúc này, trạng thái của Hệ S và Friend hòa vào nhau: (|00> + |11>) / sqrt(2)
# Đối với Friend, họ đã thấy một kết quả cụ thể. 
# Nhưng đối với Wigner (ở ngoài phòng), toàn bộ phòng thí nghiệm vẫn là một trạng thái chồng chập.

# --- Bước C: Wigner thực hiện phép đo lên toàn bộ phòng thí nghiệm ---
# Wigner đo xem Hệ S và Friend có thực sự vướng víu với nhau không bằng cổng CX và H
qc.cx(0, 2)
qc.h(0)
qc.h(1)

# --- Bước D: Thu thập kết quả ---
# Đo Friend (Lưu vào bit cổ điển 0) và Wigner (Lưu vào bit cổ điển 1)
qc.measure(1, 0) # Kết quả của Friend
qc.measure(2, 1) # Kết quả của Wigner

# =====================================================================
# 2. TÌM MÁY LƯỢNG TỬ THẬT ĐANG RẢNH NHẤT VÀ DỊCH MẠCH (TRANSPILE)
# =====================================================================
print("\nSearching for the most optimal real quantum device...")
backend = service.least_busy(simulator=False, operational=True)
print("=" * 50)
print(f"Running on real quantum hardware: {backend.name}")
print("=" * 50)

print("\nTranslating the circuit for the real quantum device...")
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(qc) 
print("Translating success")

# =====================================================================
# 3. GỬI MẠCH LÊN IBM QUANTUM VÀ CHỜ KẾT QUẢ
# =====================================================================
print("\nSending to IBM Quantum...")
sampler = SamplerV2(mode=backend) 
job = sampler.run([isa_circuit], shots=1024)

print(f"Job ID: {job.job_id()}")
print("Waiting for results from the real quantum hardware...")

result = job.result()
pub_result = result[0]

# =====================================================================
# 4. IN VÀ PHÂN TÍCH KẾT QUẢ
# =====================================================================
counts = pub_result.data.c.get_counts()
print("\n===== RESULTS FROM REAL QUANTUM HARDWARE =====")
print(counts)

In [ ]:
from qiskit import QuantumCircuit

# Khởi tạo mạch 3 qubit và 2 bit cổ điển
qc = QuantumCircuit(3, 2)

# --- Bước A: Hệ S chồng chập ---
qc.h(0) 

# --- Bước B: Friend (Qubit 1) đo Hệ S (Qubit 0) ---
qc.cx(0, 1) 

# --- Bước C: Wigner (Qubit 2) đo toàn bộ hệ thống ---
c

# --- Bước D: Ghi kết quả vào bit cổ điển ---
qc.measure(1, 0) # Kết quả của Friend -> bit 0
qc.measure(2, 1) # Kết quả của Wigner -> bit 1

# In mạch ra màn hình dưới dạng văn bản (Text-based diagram)
print("\nSơ đồ mạch mô phỏng thí nghiệm Wigner's Friend:")
print(qc.draw(output='text'))


Sơ đồ mạch mô phỏng thí nghiệm Wigner's Friend:
     ┌───┐          ┌───┐   
q_0: ┤ H ├──■────■──┤ H ├───
     └───┘┌─┴─┐  │  ├───┤┌─┐
q_1: ─────┤ X ├──┼──┤ H ├┤M├
          └───┘┌─┴─┐└┬─┬┘└╥┘
q_2: ──────────┤ X ├─┤M├──╫─
               └───┘ └╥┘  ║ 
c: 2/═════════════════╩═══╩═
                      1   0 


In [31]:
from qiskit import QuantumCircuit
import matplotlib.pyplot as plt

qc = QuantumCircuit(3, 2)

# --- Bước A: Hệ S chồng chập ---
qc.h(0)

# --- Bước B: Friend đo Hệ S ---
qc.cx(0, 1)

# --- Bước C: Wigner đo toàn bộ hệ ---
qc.cx(0, 2)
qc.h(0)
qc.h(1)

# --- Bước D: Ghi kết quả ---
qc.measure(1, 0)
qc.measure(2, 1)

print("\nSơ đồ mạch mô phỏng thí nghiệm Wigner's Friend:")

# Vẽ trực tiếp bằng matplotlib
qc.draw(output='mpl')
plt.show()


Sơ đồ mạch mô phỏng thí nghiệm Wigner's Friend:
